# 01 · Import MIMIC-IV-on-FHIR demo → WintEHR
Load the [MIMIC-IV Clinical Database Demo on FHIR (v2.1.0)](https://physionet.org/content/mimic-iv-fhir-demo/2.1.0/)
— 100 de-identified patients as ready-made FHIR R4 NDJSON — into a WintEHR HAPI server as
idempotent `PUT`-by-id transaction Bundles.

**What it handles for you:** dependency ordering (Organization → … → Observations, so HAPI's
referential integrity never trips), patient subsetting (the full demo is ~929k resources;
default imports the 5 richest patients), a provenance `meta.tag` on every imported resource
(so verification and cleanup can find exactly what this notebook wrote), and structural
validation before anything is sent.

**Resource types:** Patient, Encounter (hosp/ED/ICU), Condition, Procedure, Medication +
Request/Dispense/Administration/Statement, Specimen, Observation (labs, micro, ED vitals,
ICU flowsheet), Location, Organization.

_MIMIC dates are de-identified (shifted ~a century forward) but internally consistent —
expect encounter years like 2180._

In [1]:
import json, gzip, time, pathlib
from collections import Counter, defaultdict
import requests

# ---- Config: edit these -------------------------------------------------
DATA_DIR   = pathlib.Path.home() / "Downloads" / "mimic-iv-clinical-database-demo-on-fhir-2.1.0" / "fhir"

# WintEHR FHIR endpoints (pick one):
#   local dev stack          : http://localhost:8888/fhir        (HAPI direct)
#   wintehrdev via SSH tunnel: http://localhost:18888/fhir       (ssh -L 18888:localhost:8888 azureuser@wintehrdev...)
#   wintehrdev via app proxy : http://wintehrdev.eastus.cloudapp.azure.com/fhir
FHIR_BASE  = "http://localhost:8888/fhir"

LIMIT_PATIENTS = 5      # 5 richest patients ≈ a few 10k resources. None = all 100 (~929k, hours)
PATIENT_IDS    = []     # explicit MimicPatient UUIDs override LIMIT_PATIENTS
INCLUDE_ICU_FLOWSHEET = False  # Chartevents/Datetime/Outputevents = 75% of the dataset's volume
BATCH_SIZE     = 200    # resources per transaction Bundle

# Every imported resource gets this tag — verification + cleanup key off it.
TAG = {"system": "http://wintehr.local/fhir/import", "code": "mimic-iv-fhir-demo-2.1.0"}

# ---- Dependency-ordered file list (referential integrity holds at every step)
FILES = [
    "MimicOrganization", "MimicLocation",                      # infrastructure
    "MimicMedication", "MimicMedicationMix",                   # Mix ingredients ref Medication
    "MimicPatient",
    "MimicEncounter", "MimicEncounterED", "MimicEncounterICU", # ICU partOf hosp Encounter
    "MimicSpecimen", "MimicSpecimenLab",                       # Observations ref Specimen
    "MimicMedicationRequest",                                  # Dispense/Admin ref the Request
    "MimicMedicationDispense", "MimicMedicationDispenseED",
    "MimicMedicationAdministration", "MimicMedicationAdministrationICU",
    "MimicMedicationStatementED",
    "MimicCondition", "MimicConditionED",
    "MimicProcedure", "MimicProcedureED", "MimicProcedureICU",
    # The microbiology trio cross-references BOTH ways (Test --hasMember-->
    # Org --hasMember--> Susc, while Org/Susc --derivedFrom--> back up), so no
    # file order satisfies referential integrity. They are merged into ONE
    # group below and ingested in patient-closed transactions.
    ("Microbiology", ["MimicObservationMicroTest", "MimicObservationMicroOrg",
                      "MimicObservationMicroSusc"]),
    "MimicObservationLabevents", "MimicObservationED", "MimicObservationVitalSignsED",
]
ICU_FLOWSHEET = ["MimicObservationChartevents", "MimicObservationDatetimeevents",
                 "MimicObservationOutputevents"]
if INCLUDE_ICU_FLOWSHEET:
    FILES += ICU_FLOWSHEET

# Loaded in full regardless of patient selection (shared, tiny):
SHARED = {"MimicOrganization", "MimicLocation", "MimicMedication", "MimicMedicationMix"}

def read_ndjson(name):
    with gzip.open(DATA_DIR / f"{name}.ndjson.gz", "rt") as f:
        for line in f:
            yield json.loads(line)

print("data dir :", DATA_DIR)
print("target   :", FHIR_BASE)
print("files    :", len(FILES), "| ICU flowsheet:", "included" if INCLUDE_ICU_FLOWSHEET else "skipped")

data dir : /Users/robertbarrett/Downloads/mimic-iv-clinical-database-demo-on-fhir-2.1.0/fhir
target   : http://localhost:8888/fhir
files    : 25 | ICU flowsheet: skipped


## 1 · Inventory the dataset
Count resources per file and confirm the server is reachable before doing any work.

In [2]:
counts = {}
for entry in FILES:
    for name in (entry[1] if isinstance(entry, tuple) else [entry]):
        counts[name] = sum(1 for _ in read_ndjson(name))
        print(f"{name:38s} {counts[name]:8d}")
print(f"{'TOTAL (selected files)':38s} {sum(counts.values()):8d}")

meta = requests.get(f"{FHIR_BASE}/metadata", timeout=30).json()
print("\nserver:", meta.get("software", {}).get("name"), meta.get("software", {}).get("version"),
      "| FHIR", meta.get("fhirVersion"))

MimicOrganization                             1
MimicLocation                                31
MimicMedication                            1480
MimicMedicationMix                          314
MimicPatient                                100
MimicEncounter                              275
MimicEncounterED                            222
MimicEncounterICU                           140
MimicSpecimen                              1336
MimicSpecimenLab                          11122
MimicMedicationRequest                    17552
MimicMedicationDispense                   14293
MimicMedicationDispenseED                  1082


MimicMedicationAdministration             36131
MimicMedicationAdministrationICU          20404
MimicMedicationStatementED                 2411
MimicCondition                             4506
MimicConditionED                            545
MimicProcedure                              722
MimicProcedureED                           1260
MimicProcedureICU                          1468
MimicObservationMicroTest                  1893
MimicObservationMicroOrg                    338
MimicObservationMicroSusc                  1036


MimicObservationLabevents                107727
MimicObservationED                         2742
MimicObservationVitalSignsED               6300
TOTAL (selected files)                   235431



server: HAPI FHIR Server 8.8.0 | FHIR 4.0.1


## 2 · Select patients
The demo holds 100 patients but resource volume is heavily skewed. Rank patients by how much
clinical data they carry (encounters + meds + labs) and keep the top `LIMIT_PATIENTS` —
or set `PATIENT_IDS` explicitly. Shared resources (Organization/Location/Medication) always load.

In [3]:
patients = list(read_ndjson("MimicPatient"))
by_id = {p["id"]: p for p in patients}

# Rank by data richness over the moderate-sized files (fast full scans)
richness = Counter()
for name in ["MimicEncounter", "MimicMedicationRequest", "MimicObservationLabevents", "MimicCondition"]:
    for r in read_ndjson(name):
        ref = (r.get("subject") or {}).get("reference", "")
        if ref.startswith("Patient/"):
            richness[ref.split("/", 1)[1]] += 1

if PATIENT_IDS:
    selected = [pid for pid in PATIENT_IDS if pid in by_id]
elif LIMIT_PATIENTS:
    selected = [pid for pid, _ in richness.most_common(LIMIT_PATIENTS)]
else:
    selected = list(by_id)

sel = set(selected)
print(f"selected {len(sel)} / {len(patients)} patients:")
for pid in selected:
    p = by_id[pid]
    print(f"  {pid}  {p['name'][0]['family']:18s} {p.get('gender','?'):7s} b.{p.get('birthDate','?')}"
          f"  ~{richness[pid]} core resources")

selected 5 / 100 patients:
  cb70e6ae-90b1-562b-8ab0-467c65d18d5e  Patient_10014354   male    b.2086-10-08  ~11620 core resources
  4f773083-7f4d-5378-b839-c24ca1e15434  Patient_10035631   male    b.2049-09-17  ~11096 core resources
  91f25704-6153-5259-bdd7-2ca6478de14a  Patient_10019003   female  b.2083-12-21  ~9581 core resources
  8e77dd0b-932d-5790-9ba6-5c6df8434457  Patient_10039708   female  b.2092-10-30  ~6963 core resources
  77e10fd0-6a1c-5547-a130-fae1341acf36  Patient_10003400   female  b.2062-06-05  ~4405 core resources


## 3 · Load + filter (dependency order)
Keep a resource if it is shared infrastructure or belongs to a selected patient
(any `Patient/…` reference it carries). Each kept resource gets the provenance tag.

In [4]:
def patient_refs(o, out):
    if isinstance(o, dict):
        for k, v in o.items():
            if k == "reference" and isinstance(v, str) and v.startswith("Patient/"):
                out.add(v.split("/", 1)[1])
            else:
                patient_refs(v, out)
    elif isinstance(o, list):
        for v in o:
            patient_refs(v, out)
    return out

def tag(r):
    r.setdefault("meta", {}).setdefault("tag", []).append(dict(TAG))
    return r

batches = []   # (group_name, [resources], patient_closed) in dependency order
kept = 0
for entry in FILES:
    name, parts = entry if isinstance(entry, tuple) else (entry, [entry])
    patient_closed = isinstance(entry, tuple)   # merged groups chunk on patient boundaries
    rows = []
    for part in parts:
        for r in read_ndjson(part):
            if part in SHARED:
                rows.append(tag(r))
            elif part == "MimicPatient":
                if r["id"] in sel:
                    rows.append(tag(r))
            else:
                refs = patient_refs(r, set())
                if refs and refs <= sel:
                    r["_pt"] = next(iter(refs))   # stripped before send
                    rows.append(tag(r))
    if patient_closed:
        rows.sort(key=lambda r: r.get("_pt", ""))  # group each patient contiguously
    batches.append((name, rows, patient_closed))
    total = sum(counts[p] for p in parts)
    kept += len(rows)
    print(f"{name:38s} kept {len(rows):7d} / {total:7d}")
print(f"{'TOTAL to import':38s} {kept:12d}")

MimicOrganization                      kept       1 /       1
MimicLocation                          kept      31 /      31
MimicMedication                        kept    1480 /    1480
MimicMedicationMix                     kept     314 /     314
MimicPatient                           kept       5 /     100
MimicEncounter                         kept      54 /     275
MimicEncounterED                       kept      48 /     222
MimicEncounterICU                      kept      14 /     140
MimicSpecimen                          kept     291 /    1336
MimicSpecimenLab                       kept    3150 /   11122


MimicMedicationRequest                 kept    3900 /   17552
MimicMedicationDispense                kept    3010 /   14293
MimicMedicationDispenseED              kept     324 /    1082


MimicMedicationAdministration          kept   11103 /   36131


MimicMedicationAdministrationICU       kept    3298 /   20404
MimicMedicationStatementED             kept     886 /    2411
MimicCondition                         kept    1105 /    4506
MimicConditionED                       kept     106 /     545
MimicProcedure                         kept     122 /     722
MimicProcedureED                       kept     328 /    1260
MimicProcedureICU                      kept     149 /    1468
Microbiology                           kept     537 /    3267


MimicObservationLabevents              kept   38606 /  107727
MimicObservationED                     kept     704 /    2742
MimicObservationVitalSignsED           kept    1640 /    6300
TOTAL to import                               71206


## 4 · Structural validation
No external deps: unique type/id pairs, and every internal reference to a type we import
must resolve within the import set (references we don't import at all are listed, not fatal).

In [5]:
def all_refs(o, out):
    if isinstance(o, dict):
        for k, v in o.items():
            if k == "reference" and isinstance(v, str) and "/" in v:
                out.append(v)
            else:
                all_refs(v, out)
    elif isinstance(o, list):
        for v in o:
            all_refs(v, out)
    return out

ids, dupes = set(), 0
imported_types = set()
for name, rows, _pc in batches:
    for r in rows:
        key = f"{r['resourceType']}/{r['id']}"
        dupes += key in ids
        ids.add(key)
        imported_types.add(r["resourceType"])

unresolved = Counter()
for name, rows, _pc in batches:
    for r in rows:
        for ref in all_refs(r, []):
            rtype = ref.split("/", 1)[0]
            if rtype in imported_types and ref not in ids:
                unresolved[rtype] += 1

print(f"resources   : {len(ids)}   duplicates: {dupes}")
print(f"unresolved  : {sum(unresolved.values())}   {dict(unresolved) if unresolved else ''}")
print("OK" if dupes == 0 and not unresolved else "CHECK FAILURES ABOVE")

resources   : 71206   duplicates: 0
unresolved  : 0   
OK


### Why transaction Bundles and not HAPI's `$import`?
Fair question — NDJSON is exactly what FHIR bulk import consumes. Trade-offs that led here:

- **`$import` is disabled by default** on stock HAPI (`bulk_import_enabled: false`), and
  WintEHR's deployed HAPI doesn't enable it — flipping it means a config change + HAPI
  restart on a long-running server holding all the data.
- **`$import` pulls, it doesn't accept uploads**: each `input.url` must be an NDJSON URL the
  *server* can fetch, so these local files would first need staging somewhere
  HAPI-reachable.
- **No subsetting, no provenance**: bulk import takes whole files — no 5-patient selection,
  and nothing like our `meta.tag`, which is what makes §6 verification exact and §7 cleanup
  surgical.
- **Error visibility**: transactions fail loudly per-bundle with an `OperationOutcome`
  (that's how the microbiology `hasMember` cycle above was found); bulk import is an async
  job you poll, with failures buried in job diagnostics.
- **It exercises the real write path** — the same `/fhir` proxy the WintEHR app uses.

For a **full-scale load** (all 100 patients / ~929k resources, or the non-demo MIMIC-on-FHIR),
`$import` becomes the right tool: enable it in HAPI's config, stage the NDJSON on an
HTTP endpoint the container can reach, and let the server ingest at batch speed.

## 5 · Ingest — transaction Bundles, PUT-by-id
`PUT ResourceType/id` in a `transaction` Bundle = idempotent upsert: re-running the notebook
overwrites in place, never duplicates. Files go up in dependency order so HAPI's referential
integrity is satisfied at every step.

In [6]:
def put_bundle(rows):
    bundle = {"resourceType": "Bundle", "type": "transaction", "entry": [
        {"resource": r, "request": {"method": "PUT", "url": f"{r['resourceType']}/{r['id']}"}}
        for r in rows]}
    # Trailing slash matters: WintEHR's backend /fhir proxy 301s a bare POST
    # /fhir (and the redirect turns into a GET); HAPI direct accepts both.
    resp = requests.post(FHIR_BASE.rstrip("/") + "/", json=bundle,
                         headers={"Content-Type": "application/fhir+json"}, timeout=300)
    if resp.status_code not in (200, 201):
        raise RuntimeError(f"HTTP {resp.status_code}: {resp.text[:800]}")
    return resp.json()

def chunks_of(rows, patient_closed):
    # <= BATCH_SIZE per bundle; patient-closed groups never split a patient
    # (their cross-references must resolve inside one transaction).
    if not patient_closed:
        for i in range(0, len(rows), BATCH_SIZE):
            yield rows[i:i + BATCH_SIZE]
        return
    chunk, cur = [], None
    for r in rows:
        if len(chunk) >= BATCH_SIZE and r.get("_pt") != cur:
            yield chunk; chunk = []
        cur = r.get("_pt")
        chunk.append(r)
    if chunk:
        yield chunk

t0, sent = time.time(), 0
for name, rows, patient_closed in batches:
    if not rows:
        continue
    for chunk in chunks_of(rows, patient_closed):
        for r in chunk:
            r.pop("_pt", None)   # internal bookkeeping, not FHIR
        put_bundle(chunk)
        sent += len(chunk)
    print(f"{name:38s} {len(rows):7d} sent   ({sent}/{kept}, {time.time()-t0:5.0f}s)")
print(f"\ndone: {sent} resources in {time.time()-t0:.0f}s")

MimicOrganization                            1 sent   (1/71206,     0s)


MimicLocation                               31 sent   (32/71206,     0s)


MimicMedication                           1480 sent   (1512/71206,     4s)


MimicMedicationMix                         314 sent   (1826/71206,     5s)
MimicPatient                                 5 sent   (1831/71206,     5s)


MimicEncounter                              54 sent   (1885/71206,     6s)


...

MimicProcedureICU                          149 sent   (29719/71206,    75s)


Microbiology                               537 sent   (30256/71206,    76s)


MimicObservationLabevents                38606 sent   (68862/71206,   199s)


MimicObservationED                         704 sent   (69566/71206,   201s)


MimicObservationVitalSignsED              1640 sent   (71206/71206,   205s)

done: 71206 resources in 205s


## 6 · Verify
Count what the server now holds under this import's tag, and spot-check one patient's record.

In [7]:
tag_param = f"{TAG['system']}|{TAG['code']}"
print(f"{'type':24s} {'imported':>9s} {'on server':>10s}")
mismatch = False
for rtype in sorted(imported_types):
    local = sum(1 for _ in ids if _.startswith(rtype + "/"))
    r = requests.get(f"{FHIR_BASE}/{rtype}", params={"_tag": tag_param, "_summary": "count"}, timeout=60)
    server = r.json().get("total", "?")
    flag = "" if server == local else "  <-- MISMATCH"
    mismatch |= bool(flag)
    print(f"{rtype:24s} {local:9d} {server:>10} {flag}")

# Spot-check the richest patient with the same reads the WintEHR UI makes
pid = selected[0]
fam = by_id[pid]["name"][0]["family"]
pt  = requests.get(f"{FHIR_BASE}/Patient/{pid}", timeout=30)
by_name = requests.get(f"{FHIR_BASE}/Patient", params={"name": fam, "_summary": "count"}, timeout=30).json()
encs = requests.get(f"{FHIR_BASE}/Encounter", params={"patient": pid, "_summary": "count"}, timeout=60).json()
labs = requests.get(f"{FHIR_BASE}/Observation",
                    params={"patient": pid, "category": "laboratory", "_summary": "count"}, timeout=60).json()
print(f"\nspot-check {fam}:")
print(f"  Patient read      HTTP {pt.status_code}")
print(f"  search by name    {by_name.get('total')} match")
print(f"  Encounters        {encs.get('total')}")
print(f"  lab Observations  {labs.get('total')}")
print(f"\nIn the WintEHR UI: log in and search the patient list for {fam!r}.")
print("\nALL COUNTS MATCH" if not mismatch else "SOME COUNTS MISMATCH -- see above")

type                      imported  on server


Condition                     1211       1211 


Encounter                      116        116 
Location                        31         31 


Medication                    1794       1794 


MedicationAdministration     14401      14401 
MedicationDispense            3334       3334 


MedicationRequest             3900       3900 
MedicationStatement            886        886 


Observation                  41487      41487 
Organization                     1          1 


Patient                          5          5 


Procedure                      599        599 
Specimen                      3441       3441 



spot-check Patient_10014354:
  Patient read      HTTP 200
  search by name    1 match
  Encounters        47
  lab Observations  9789

In the WintEHR UI: log in and search the patient list for 'Patient_10014354'.

ALL COUNTS MATCH


## 7 · (Optional) remove this import
Deletes exactly the resources this notebook tagged, in **reverse** dependency order (children
before the Patients/Medications they reference). Guarded — flip `DO_CLEANUP` deliberately.

In [8]:
DO_CLEANUP = False

if DO_CLEANUP:
    deleted = 0
    for name, rows, _pc in reversed(batches):
        for i in range(0, len(rows), BATCH_SIZE):
            chunk = rows[i:i + BATCH_SIZE]
            bundle = {"resourceType": "Bundle", "type": "transaction", "entry": [
                {"request": {"method": "DELETE", "url": f"{r['resourceType']}/{r['id']}"}}
                for r in chunk]}
            requests.post(FHIR_BASE.rstrip("/") + "/", json=bundle,
                          headers={"Content-Type": "application/fhir+json"}, timeout=300).raise_for_status()
            deleted += len(chunk)
        if rows:
            print(f"{name:38s} deleted {len(rows)}")
    print("total deleted:", deleted)
else:
    print("Cleanup disabled. Set DO_CLEANUP = True and re-run this cell to remove the import.")

Cleanup disabled. Set DO_CLEANUP = True and re-run this cell to remove the import.
